# Derived Rasters — Visual QA

Inspect pre-computed rasters in `work/derived_rasters/`.  
All rasters are CONUS-scale (~124k × 167k px at 30 m).  
**Best practice:** uses rasterio's `out_shape` decimated read — only a thumbnail-resolution
array is decompressed; if overviews exist, GDAL uses them automatically.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.enums import Resampling

DATA_ROOT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2_param_v2")
DERIVED_DIR = DATA_ROOT / "work" / "derived_rasters"

# Target display width in pixels — all reads are decimated to this
DISPLAY_PX = 2000


## Available rasters

In [ ]:
SKIP = {"intermediate"}

for p in sorted(DERIVED_DIR.glob("*.tif")):
    if any(s in p.stem for s in SKIP):
        continue
    ok = False
    try:
        with rasterio.open(p):
            ok = True
    except Exception:
        pass
    status = "✓" if ok else "✗ (corrupt/partial)"
    size_mb = round(p.stat().st_size / 1024**2)
    print(f"  {status}  {size_mb:>6} MB  {p.name}")


## Helper functions

In [ ]:

def raster_meta(path: Path) -> dict:
    """Return metadata for a raster without loading pixel data."""
    with rasterio.open(path) as src:
        return {
            "shape": src.shape,
            "dtype": src.dtypes[0],
            "crs": src.crs,
            "bounds": src.bounds,
            "res_m": abs(src.transform.a),
            "nodata": src.nodata,
            "overviews": src.overviews(1),
            "size_MB": round(path.stat().st_size / 1024**2),
        }


def decimated_read(path: Path, target_px: int = DISPLAY_PX, resampling=Resampling.average):
    """
    Read a decimated thumbnail from a large raster via rasterio out_shape.

    Only the required pixels are decompressed (GDAL decimated read).
    If internal overviews are present, GDAL selects the best one automatically.
    Returns (masked_array_2d, scaled_transform, meta_dict).
    """
    with rasterio.open(path) as src:
        factor = max(1, max(src.width, src.height) // target_px)
        out_h = max(1, src.height // factor)
        out_w = max(1, src.width // factor)
        data = src.read(
            1,
            out_shape=(out_h, out_w),
            resampling=resampling,
        ).astype(np.float64)
        nodata = src.nodata
        t = src.transform
        scaled_transform = t * t.scale(src.width / out_w, src.height / out_h)
        meta = raster_meta(path)

    mask = ~np.isfinite(data)
    if nodata is not None:
        mask |= data == nodata
    return np.ma.array(data, mask=mask), scaled_transform, meta


def percentile_stretch(arr, lo=2, hi=98):
    valid = arr.compressed() if isinstance(arr, np.ma.MaskedArray) else arr[np.isfinite(arr)]
    if valid.size == 0:
        return arr, 0.0, 1.0
    vmin, vmax = np.percentile(valid, [lo, hi])
    return np.clip(arr, vmin, vmax), vmin, vmax


def plot_raster(path: Path, title: str, cmap: str = "viridis", units: str = "",
                stretch_pct: tuple = (2, 98)):
    """Display a large raster as a decimated overview with a distribution histogram."""
    if not path.exists():
        print(f"  Skipping {path.name} — not found yet")
        return
    try:
        data, _, meta = decimated_read(path)
    except Exception as e:
        print(f"  Cannot read {path.name}: {e}")
        return

    stretched, vmin, vmax = percentile_stretch(data, lo=stretch_pct[0], hi=stretch_pct[1])
    valid = data.compressed()

    if valid.size == 0:
        print(f"  {path.name} — thumbnail entirely masked.")
        print("  Likely cause: nodata=NaN not declared in file; rebuild rasters first.")
        plt.close()
        return

    p25, p50, p75 = np.percentile(valid, [25, 50, 75])
    h, w = meta["shape"]

    fig, (ax_img, ax_hist) = plt.subplots(
        1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 1]}
    )

    im = ax_img.imshow(stretched, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax_img.set_title(f"{title}\n{path.name}  ({meta['size_MB']} MB)", fontsize=10)
    ax_img.axis("off")
    plt.colorbar(im, ax=ax_img, fraction=0.03, pad=0.02, label=units)

    ax_hist.hist(valid, bins=150, color="steelblue", edgecolor="none", density=True)
    ax_hist.axvline(p25,  color="cornflowerblue", linestyle="--", linewidth=1.2, label=f"p25 = {p25:.3g}")
    ax_hist.axvline(p50,  color="navy",           linestyle="-",  linewidth=1.8, label=f"p50 = {p50:.3g}")
    ax_hist.axvline(p75,  color="cornflowerblue", linestyle="--", linewidth=1.2, label=f"p75 = {p75:.3g}")
    ax_hist.axvline(vmin, color="orange",         linestyle=":",  linewidth=1.2, label=f"stretch lo = {vmin:.3g}")
    ax_hist.axvline(vmax, color="red",            linestyle=":",  linewidth=1.2, label=f"stretch hi = {vmax:.3g}")
    ax_hist.set_title(
        f"{h:,} \u00d7 {w:,} px  |  {meta['res_m']:.0f} m\n"
        f"valid: {valid.size:,}  mean: {valid.mean():.4g}  std: {valid.std():.4g}",
        fontsize=8, loc="left", family="monospace",
    )
    ax_hist.set_xlabel(units or "value")
    ax_hist.set_ylabel("density")
    ax_hist.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


## Resampled Root Depth (`rd_250_raw.tif`)

Root zone depth resampled from 250 m to match the AWC (available water capacity) grid.

- **Source:** Global/continental root depth dataset at 250 m native resolution (`input/lulc_veg/RootDepth.tif`)
- **Processing:** `resample()` — GDAL nearest-neighbour reproject to AWC extent/CRS, then `mask_values=(128, 0)` and `mask_negative=True` applied block-wise before writing the final float64 GeoTIFF
- **Units:** cm — depth from soil surface to the bottom of the effective root zone
- **Expected range:** ~10–200 cm; shallow-rooted crops/grasses ≈ 20–60 cm, deep-rooted forests ≈ 80–200 cm
- **Role in model:** Multiplied by AWC to produce `soil_moist_max`; larger values → greater soil-zone storage capacity in PRMS/NHM

### ⚠️ White areas (masked pixels) in Plains / Central Valley / East

White pixels are **NoData** (masked array entries rendered as white by matplotlib).  
Their origin: the source `RootDepth.tif` uses **0** to encode agricultural/cultivated land
(annual crops — wheat, corn, soy in the Great Plains; irrigated crops in the CA Central Valley;
row crops in the Midwest/East) and possibly impervious/water pixels.
`resample()` converts 0 → NaN via `mask_values=(128, 0)`, so these areas drop out entirely.

**Consequences:**
- `soil_moist_max` inherits the same holes (multiplying NaN × AWC = NaN)
- HRUs dominated by cropland will have no root-depth signal and may receive NaN or a fallback value during the zonal-statistics step
- **A gap-fill step (e.g., assign a default crop rooting depth ~30–50 cm) is needed before these parameters are used in model calibration/runs**


In [ ]:
plot_raster(DERIVED_DIR / "rd_250_raw.tif", "Resampled Root Depth", cmap="YlOrBr", units="cm")


## Soil Moisture Max (`soil_moist_max.tif`)

Maximum plant-available soil moisture storage in the root zone: **RootDepth (cm) × AWC (cm/cm)**.

- **Source:** Derived product — `rd_250_raw × AWC` computed pixel-by-pixel
- **Units:** cm of plant-available water stored in the root zone
- **Expected range:** ~5–80 cm; low values in rocky/shallow or sandy soils, high values in deep loamy soils
- **Distribution:** Typically right-skewed; urban/impervious and bare/rocky areas cluster near zero, humid forest soils extend the upper tail
- **Role in model:** Primary soil-zone capacity parameter (`soil_moist_max`) in PRMS/NHM; larger values slow baseflow recession and buffer streamflow


In [ ]:
plot_raster(DERIVED_DIR / "soil_moist_max.tif", "Soil Moisture Max (RootDepth × AWC)", cmap="Blues", units="cm")


## Resampled Canopy Cover (`cnpy_resampled_lulc.tif`)

Tree canopy cover fraction resampled from its native resolution to match the LULC raster.

- **Source:** `input/lulc_veg/nhm_v11/CNPY.tif` — int16, nodata=−32768, shape 110020×155443
- **Template:** `input/lulc_veg/nhm_v11/LULC.tif` — shape 124131×166734 (~12% larger in each dimension)
- **Units:** % cover (0–100)
- **Downstream use:** Multiplied by `keep` to produce `radtrn`

In [ ]:
plot_raster(DERIVED_DIR / "cnpy_resampled_lulc.tif", "Canopy Cover (resampled to LULC grid)", cmap="Greens", units="%")



## Resampled Winter Retention (`keep_resampled_lulc.tif`)

Fraction of canopy retained through the dormant/winter season, resampled to match the LULC grid.

- **Source:** Lookup-table estimate derived from LULC classes: evergreen = ~100 %, deciduous = ~0 %, mixed/shrub = intermediate
- **Processing:** Per-pixel values assigned from LULC class, then resampled to the LULC grid extent
- **Units:** % (0–100); 0 = fully deciduous (bare canopy in winter), 100 = fully evergreen
- **Expected distribution:** Strongly bimodal — a dominant peak near 0 (deciduous forest, cropland, non-forest) and a secondary peak near 100 (evergreen conifers); intermediate values for mixed and shrub classes
- **Role in model:** Scales the canopy cover seasonally; combined with `cnpy` to compute annual-average radiation transmission (`radtrn = cnpy × keep / 100`)


In [ ]:
plot_raster(DERIVED_DIR / "keep_resampled_lulc.tif", "Winter Retention (resampled to LULC grid)", cmap="PuBuGn", units="%")


## Radiation Transmission (`radtrn_nhm_v11.tif`)

Annual-average fraction of incoming shortwave radiation **transmitted through** the canopy to the surface.

- **Formula:** `cnpy × keep / 100` where LULC class ≥ 3 (tree classes); 0 elsewhere
- **Units:** Dimensionless fraction (0–1); 0 = completely open sky, 1 = fully blocked by closed evergreen canopy
- **Expected distribution:** Highly zero-inflated — the large non-forested fraction of CONUS produces a dominant mass at 0; a long right tail extends to ~0.8–0.9 for dense evergreen forest
- **Note:** Values above ~0.7 represent extremely dense closed-canopy stands and should be compared against the source `cnpy` and `keep` layers if they appear unexpectedly high
- **Role in model:** NHM v1.1 `rad_trncf` parameter — attenuates solar radiation reaching the sub-canopy snowpack and soil; higher values reduce snow-melt rates and sub-canopy ET


In [ ]:
plot_raster(DERIVED_DIR / "radtrn_nhm_v11.tif", "Radiation Transmission (NHM v1.1)", cmap="plasma", units="fraction (0–1)")


## Side-by-side: canopy, keep, radtrn

In [ ]:
targets = [
    ("cnpy_resampled_lulc.tif",  "Canopy (%)",           "Greens"),
    ("keep_resampled_lulc.tif",  "Winter Retention (%)", "PuBuGn"),
    ("radtrn_nhm_v11.tif",       "Rad. Transmission",    "plasma"),
]

available = [(fn, lbl, cm) for fn, lbl, cm in targets if (DERIVED_DIR / fn).exists()]
if not available:
    print("None of the comparison rasters are ready yet.")
else:
    fig, axes = plt.subplots(1, len(available), figsize=(7 * len(available), 6))
    if len(available) == 1:
        axes = [axes]
    for ax, (fn, label, cmap) in zip(axes, available):
        try:
            data, _, _ = decimated_read(DERIVED_DIR / fn)
            stretched, vmin, vmax = percentile_stretch(data)
            im = ax.imshow(stretched, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
            ax.set_title(label)
            ax.axis("off")
            plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        except Exception as e:
            ax.set_title(f"{label}\n(error: {e})")
            ax.axis("off")
    plt.suptitle("NHM v1.1 LULC-derived rasters — decimated overview", fontsize=13)
    plt.tight_layout()
    plt.show()
